# Impulse - A2D2 Object Tracking (ByteTrack)

Fourth notebook of the A2D2 demo. Applies **ByteTrack** object tracking to the detection
staging table produced by `a2d2_object_detection.ipynb`, assigning persistent track IDs
across frames.

Each tracked object receives a globally unique **UUID** (`entity_id`) derived deterministically
from `(container_id, channel_name, track_id)` using UUID5 - ensuring stable IDs across re-runs.

Runs on **serverless** (supervision/cv2 works in serverless but not on classic ML clusters).

## Prerequisite
Run `a2d2_object_detection.ipynb` first to populate the staging table with bounding boxes.

In [ ]:
%pip install -U supervision
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")
dbutils.widgets.text("table_prefix", "a2d2", "Table Prefix")
dbutils.widgets.text("volume", "a2d2_raw", "UC Volume")
# ByteTrack parameters
dbutils.widgets.text("track_thresh", "0.5", "Track confidence threshold")
dbutils.widgets.text("match_thresh", "0.8", "IoU matching threshold")
dbutils.widgets.text("track_buffer", "30", "Frames to keep lost tracks")

In [ ]:
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
TABLE_PREFIX = dbutils.widgets.get("table_prefix")
VOLUME = dbutils.widgets.get("volume")
TRACK_THRESH = float(dbutils.widgets.get("track_thresh") or "0.5")
MATCH_THRESH = float(dbutils.widgets.get("match_thresh") or "0.8")
TRACK_BUFFER = int(dbutils.widgets.get("track_buffer") or "30")

if not (CATALOG and SCHEMA and TABLE_PREFIX):
    raise ValueError("Please set catalog, schema, and table_prefix widgets above.")

pfx = f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}"
STAGING = f"{pfx}_detect_staging"
TRACKING = f"{pfx}_tracking_fact"

print(f"ByteTrack: track_thresh={TRACK_THRESH}, match_thresh={MATCH_THRESH}, track_buffer={TRACK_BUFFER}")
print(f"Reading from: {STAGING}")
print(f"Writing to: {TRACKING}")

## ByteTrack Tracking

ByteTrack tracks objects across frames using a two-stage association:
1. High-confidence detections are matched first via IoU
2. Low-confidence detections are then matched to remaining tracks

This approach recovers objects that are temporarily occluded or have lower detection scores.
We use `applyInPandas` to process each (container_id, channel_name) group independently.

In [ ]:
import pyspark.sql.types as T

TRACKING_SCHEMA = T.StructType([
    T.StructField("container_id",  T.LongType(),    False),
    T.StructField("cam_tstamp_us", T.LongType(),    False),
    T.StructField("channel_name",  T.StringType(),  False),
    T.StructField("entity_id",     T.StringType(),  False),
    T.StructField("x1",            T.DoubleType(),  False),
    T.StructField("y1",            T.DoubleType(),  False),
    T.StructField("x2",            T.DoubleType(),  False),
    T.StructField("y2",            T.DoubleType(),  False),
    T.StructField("score",         T.DoubleType(),  False),
])

def track_container_channel(pdf):
    """Apply ByteTrack to one (container_id, channel_name) group.
    
    Each tracked object gets a deterministic UUID based on (container_id, channel_name, track_id)
    so entity IDs are globally unique and stable across re-runs.
    """
    import supervision as sv
    import numpy as np
    import pandas as pd
    import uuid

    tracker = sv.ByteTrack(
        track_activation_threshold=TRACK_THRESH,
        minimum_matching_threshold=MATCH_THRESH,
        lost_track_buffer=TRACK_BUFFER,
        minimum_consecutive_frames=1,
    )
    results = []
    pdf_sorted = pdf.sort_values("cam_tstamp_us")
    entity_ids = {}  # ByteTrack track_id -> UUID

    for _, row in pdf_sorted.iterrows():
        cid = row["container_id"]
        ts = row["cam_tstamp_us"]
        ch = row["channel_name"]
        boxes = row["boxes"]

        if boxes is None or len(boxes) == 0:
            continue

        xyxy = np.array([[b["x1"], b["y1"], b["x2"], b["y2"]] for b in boxes])
        confidence = np.array([b["score"] for b in boxes])
        detections = sv.Detections(xyxy=xyxy, confidence=confidence)
        tracked = tracker.update_with_detections(detections)

        for i in range(len(tracked)):
            tid = int(tracked.tracker_id[i])
            if tid not in entity_ids:
                # Deterministic UUID from (container_id, channel_name, track_id)
                entity_ids[tid] = str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{cid}:{ch}:{tid}"))
            results.append({
                "container_id": int(cid),
                "cam_tstamp_us": int(ts),
                "channel_name": ch,
                "entity_id": entity_ids[tid],
                "x1": float(tracked.xyxy[i][0]),
                "y1": float(tracked.xyxy[i][1]),
                "x2": float(tracked.xyxy[i][2]),
                "y2": float(tracked.xyxy[i][3]),
                "score": float(tracked.confidence[i]),
            })

    if results:
        return pd.DataFrame(results)
    else:
        return pd.DataFrame(
            columns=["container_id", "cam_tstamp_us", "channel_name",
                     "entity_id", "x1", "y1", "x2", "y2", "score"])

In [ ]:
# Read staging table (only rows with bounding boxes)
staging_df = (spark.read.table(STAGING)
    .where("boxes IS NOT NULL")
    .where("channel_name LIKE 'detected_%' OR channel_name LIKE 'ahead_%'"))

n_rows = staging_df.count()
print(f"Staging rows with boxes: {n_rows}")

if n_rows == 0:
    print("No detections to track - ensure a2d2_object_detection.ipynb ran first.")
else:
    # Apply tracking per (container_id, channel_name) group
    tracking_df = (staging_df
        .groupBy("container_id", "channel_name")
        .applyInPandas(track_container_channel, schema=TRACKING_SCHEMA))
    
    # Write tracking results
    tracking_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TRACKING)
    print(f"Tracking results written to {TRACKING}")

## Summary

In [ ]:
import pyspark.sql.functions as F

tracking_result = spark.read.table(TRACKING)
total_detections = tracking_result.count()
unique_entities = tracking_result.select("entity_id").distinct().count()

print(f"Total tracked detections: {total_detections}")
print(f"Unique tracked entities: {unique_entities}")

print("\nPer-channel summary:")
display(tracking_result.groupBy("container_id", "channel_name")
        .agg(F.count("*").alias("detections"),
             F.countDistinct("entity_id").alias("unique_entities"))
        .orderBy("container_id", "channel_name"))